In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [2]:
dataset = pd.read_csv('CKD.csv')

In [3]:
dataset=pd.get_dummies(dataset,drop_first=True)

In [4]:
indep=dataset[['age', 'bp', 'al', 'su', 'bgr', 'bu', 'sc', 'sod', 'pot', 'hrmo', 'pcv','wc', 'rc', 'sg_b', 'sg_c', 'sg_d', 'sg_e', 'rbc_normal', 'pc_normal','pcc_present', 'ba_present', 'htn_yes', 'dm_yes', 'cad_yes','appet_yes', 'pe_yes', 'ane_yes']]
dep=dataset['classification_yes']

In [5]:
#split into training set and test
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(indep, dep, test_size = 1/3, random_state = 0)


In [6]:
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

In [7]:
from sklearn.ensemble import RandomForestClassifier

In [8]:
from sklearn.model_selection import GridSearchCV

param_grid = {'criterion':['gini','entropy'],
              'max_features': ['auto','sqrt','log2'],
              'n_estimators':[10,100]} 

grid = GridSearchCV(RandomForestClassifier(), param_grid,cv=5, refit = True, verbose = 3,n_jobs=-1,scoring='f1_weighted') 
   
grid.fit(X_train, y_train) 
 

Fitting 5 folds for each of 12 candidates, totalling 60 fits


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done  16 tasks      | elapsed:    3.7s
[Parallel(n_jobs=-1)]: Done  60 out of  60 | elapsed:    4.6s finished


GridSearchCV(cv=5, estimator=RandomForestClassifier(), n_jobs=-1,
             param_grid={'criterion': ['gini', 'entropy'],
                         'max_features': ['auto', 'sqrt', 'log2'],
                         'n_estimators': [10, 100]},
             scoring='f1_weighted', verbose=3)

In [9]:
re=grid.cv_results_

grid_predictions = grid.predict(X_test) 
   
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, grid_predictions)
 
from sklearn.metrics import classification_report
clf_report = classification_report(y_test, grid_predictions)

In [10]:
from sklearn.metrics import f1_score
f1_macro=f1_score(y_test,grid_predictions,average='weighted')
print("The f1_macro value for best parameter {}:".format(grid.best_params_),f1_macro)

The f1_macro value for best parameter {'criterion': 'gini', 'max_features': 'log2', 'n_estimators': 100}: 0.9849624060150376


In [11]:
print("The confusion Matrix:\n",cm)

The confusion Matrix:
 [[50  1]
 [ 1 81]]


In [12]:
print("The report:\n",clf_report)

The report:
               precision    recall  f1-score   support

           0       0.98      0.98      0.98        51
           1       0.99      0.99      0.99        82

    accuracy                           0.98       133
   macro avg       0.98      0.98      0.98       133
weighted avg       0.98      0.98      0.98       133



In [13]:
from sklearn.metrics import roc_auc_score

roc_auc_score(y_test,grid.predict_proba(X_test)[:,1])


0.9997608799617408

In [14]:
table=pd.DataFrame.from_dict(re)

In [15]:
table

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_criterion,param_max_features,param_n_estimators,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.030919,0.001260,0.003989,9.488940e-07,gini,auto,10,"{'criterion': 'gini', 'max_features': 'auto', ...",0.945100,1.000000,0.981217,0.981031,0.981031,0.977676,0.017858,3
1,0.266886,0.004661,0.017952,1.092841e-03,gini,auto,100,"{'criterion': 'gini', 'max_features': 'auto', ...",1.000000,0.942166,0.981217,0.962264,1.000000,0.977129,0.022389,5
2,0.029527,0.001483,0.003984,6.310006e-04,gini,sqrt,10,"{'criterion': 'gini', 'max_features': 'sqrt', ...",0.981569,0.961755,0.944023,0.962264,0.981031,0.966128,0.014023,10
3,0.261901,0.003868,0.018550,2.240274e-03,gini,sqrt,100,"{'criterion': 'gini', 'max_features': 'sqrt', ...",1.000000,0.961755,0.944023,0.981031,0.981031,0.973568,0.019092,7
4,0.027925,0.001410,0.003590,4.890458e-04,gini,log2,10,"{'criterion': 'gini', 'max_features': 'log2', ...",1.000000,0.961755,0.981217,0.962264,0.962264,0.973500,0.015180,8
5,0.264095,0.007007,0.017353,4.886558e-04,gini,log2,100,"{'criterion': 'gini', 'max_features': 'log2', ...",1.000000,0.961755,0.981217,0.981031,1.000000,0.984801,0.014284,1
6,0.027927,0.001094,0.003590,4.878012e-04,entropy,auto,10,"{'criterion': 'entropy', 'max_features': 'auto...",1.000000,0.981014,0.925524,0.981031,1.000000,0.977514,0.027345,4
7,0.268881,0.010104,0.017354,7.981779e-04,entropy,auto,100,"{'criterion': 'entropy', 'max_features': 'auto...",1.000000,0.961755,0.944023,0.962264,1.000000,0.973608,0.022528,6
8,0.028524,0.001017,0.003790,3.995184e-04,entropy,sqrt,10,"{'criterion': 'entropy', 'max_features': 'sqrt...",0.981569,0.961755,0.888515,0.962264,1.000000,0.958821,0.037886,11
9,0.257113,0.005997,0.016156,1.716011e-03,entropy,sqrt,100,"{'criterion': 'entropy', 'max_features': 'sqrt...",1.000000,0.942166,0.944023,0.981031,1.000000,0.973444,0.025737,9


In [16]:
age_input=float(input("Age:"))
bp_input=float(input("bp:"))
al_input=float(input("al:"))
su_input=float(input("su:"))
bgr_input=float(input("bgr:"))
bu_input=float(input("bu:"))
sc_input=float(input("sc:"))
sod_input=float(input("sod:"))
pot_input=float(input("pot:"))
hrmo_input=float(input("hrmo:"))
pcv_input=float(input("pcv:"))
wc_input=float(input("wc:"))
rc_input=float(input("rc:"))
sg_b_input=float(input("sg_b yes 0 or 1:"))
sg_c_input=float(input("sg_c Yes 0 or 1:"))
sg_d_input=float(input("sg_d Yes 0 or 1:"))
sg_e_input=float(input("sg_e Yes 0 or 1:"))
rbc_normal_input=float(input("rbc Normal 0 or 1:"))
pc_normal_input=float(input("pc Normal 0 or 1:"))
pcc_present_input=float(input("pcc Present 0 or 1:"))
ba_present_input=float(input("ba Present 0 or 1:"))
htn_yes_input=float(input("htn Yes 0 or 1:"))
dm_yes_input=float(input("dm Yes 0 or 1:"))
cad_yes_input=float(input("cad Yes 0 or 1:"))
appet_yes_input=float(input("appet Yes 0 or 1:"))
pe_yes_input=float(input("pe Good 0 or 1:"))
ane_yes_input=float(input("ane Yes 0 or 1:"))

Age:22
bp:60
al:0
su:0
bgr:97
bu:18
sc:1.2
sod:138
pot:4.3
hrmo:13
pcv:42
wc:7900
rc:6
sg_b yes 0 or 1:1
sg_c Yes 0 or 1:0
sg_d Yes 0 or 1:0
sg_e Yes 0 or 1:0
rbc Normal 0 or 1:1
pc Normal 0 or 1:1
pcc Present 0 or 1:0
ba Present 0 or 1:0
htn Yes 0 or 1:0
dm Yes 0 or 1:0
cad Yes 0 or 1:0
appet Yes 0 or 1:0
pe Good 0 or 1:0
ane Yes 0 or 1:0


In [17]:
Future_Prediction=grid.predict([[age_input,bp_input,al_input,su_input,bgr_input,bu_input,sc_input,sod_input,pot_input,hrmo_input,pcv_input,wc_input,rc_input,sg_b_input,sg_c_input,sg_d_input,sg_e_input,rbc_normal_input,pc_normal_input,pcc_present_input,ba_present_input,htn_yes_input,dm_yes_input,cad_yes_input,appet_yes_input,pe_yes_input,ane_yes_input]])
print("Future_Prediction={}".format(Future_Prediction))

Future_Prediction=[1]
